# Tune beam current and trajectory through RFQ

In [1]:
import time
import datetime
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import OrderedDict
# from epics import caput, caget, caget_many
from epics import caget, caget_many

In [2]:
import sys
repo_root = '/user/shared/pkgs/stBO'
sys.path.insert(0, str(repo_root))
from stbo.optimization import BOController
from stbo.utils import live_monitor_plot, live_history_plot

In [3]:
repo_root = '/user/shared/pkgs/machineIO'
sys.path.insert(0, str(repo_root))
from machineIO import construct_machineIO, Evaluator, OracleEvaluator
from machineIO.objFunc import SingleTaskObjectiveFunction
from machineIO import preset

# Prepare and Check machine status

##### check source and chopper setting

In [4]:
SCS = caget("ACS_DIAG:DEST:ACTIVE_ION_SOURCE")
ion = caget("FE_ISRC"+str(SCS)+":BEAM:ELMT_BOOK")
Q = caget("FE_ISRC"+str(SCS)+":BEAM:Q_BOOK")
A = caget("FE_ISRC"+str(SCS)+":BEAM:A_BOOK")
# AQ = caget("FE_ISRC2:BEAM:MOVRQ_BOOK")
AQ = A/Q
ion = str(A)+ion+str(Q)
print('SCS'+str(SCS), ion, 'A/Q=',AQ)

duty = np.round(caget('GTS_FTS:MSTR_N0001:PCUR_DFAC_RD'),decimals=4)
mode = caget('GTS_FTS:MSTR_N0001:MSG_RD_FSM')
rep = caget('GTS_FTS:MSTR_N0001:FR_CSET_REP')

print(f'Mode: {mode}')
print(f'Rep Rate: {rep}Hz, Duty: {duty}%')

SCS2 238U37 A/Q= 6.4324324324324325
Mode: O8.1 FE Commissioning 100Hz
Rep Rate: 100.0Hz, Duty: 25.0%


##### check upstream FC

In [5]:
if SCS==1:
    upstream_FC_isoutPVs  = ['FE_SCS1:FC_D0739:LMPOS_RSTS_DRV']
    upstream_FC_insertPVs = ['FE_SCS1:FC_D0738:IN_CMD_DRV']
elif SCS==2:
    upstream_FC_isoutPVs  = ['FE_SCS2:FC_D0717:LMPOS_RSTS_DRV']
    upstream_FC_insertPVs = ['FE_SCS2:FC_D0717:IN_CMD_DRV']
    
upstream_FC_isoutPVs = upstream_FC_isoutPVs + ['FE_LEBT:FC_D0814:LMPOS_RSTS_DRV','FE_LEBT:FC_D0977:LMPOS_RSTS_DRV','FE_LEBT:FC_D0998:LMOUT_RSTS']
upstream_FC_insertPVs= upstream_FC_insertPVs+ ['FE_LEBT:FC_D0814:IN_CMD_DRV', 'FE_LEBT:FC_D0977:IN_CMD_DRV', 'FE_LEBT:FC_D0998:IN_CMD']
    
isCMDsent = False
for FC_isoutPV, FC_insertPV in zip(upstream_FC_isoutPVs, upstream_FC_insertPVs):
    is_FC_out = caget(FC_isoutPV)
    if not is_FC_out:
        r = input(f"FC {FC_insertPV.strip(':IN_CMD')} is inserted. Do you allow me to take it out? ")
        if r in ['yes','y','YES','Y']:
            caput(FC_insertPV,0)
            isCMDsent = True
if isCMDsent:
    time.sleep(10)
for FC_isoutPV, FC_insertPV in zip(upstream_FC_isoutPVs, upstream_FC_insertPVs):
    is_FC_out = caget(FC_isoutPV)
    if not is_FC_out:
        r = input(f"FC {FC_insertPV.strip(':IN_CMD')} is still inserted. please check manually. enter to continue")

##### check MEBT FC

In [6]:
is_FC_out = caget('FE_MEBT:FC_D1102:LMOUT_RSTS')
if is_FC_out:
    r = input("FE_MEBT:FC_D1102 is out. Do you allow me to insert it? ")
    if r in ['yes','y','YES','Y']:
        caput('FE_MEBT:FC_D1102:IN_CMD',1)
        time.sleep(5)
        is_FC_out = caget('FE_MEBT:FC_D1102:LMOUT_RSTS')
        if is_FC_out:
            r = input(f"FE_MEBT:FC_D1102 is still not inserted. please check manually. enter to continue")

# check MEBT FC range 1055 uA
if caget('FE_MEBT:FC_D1102:RNG_CMD') != 0:
    r = input("FE_MEBT:FC_D1102 range is set to 1uA. Do you want me to change too 1055 uA? ")
    if r in ['yes','y','YES','Y']:
        caput('FE_MEBT:FC_D1102:RNG_CMD',0)
        time.sleep(5)
        if caget('FE_MEBT:FC_D1102:RNG_CMD') != 0:
            r = input(f"FE_MEBT:FC_D1102 range is still not correct. please check manually.")
            
# check Pico vs TetrA
is_tetra = caget('DIAG-RIO01:TETRAMM_ENABLED1')
if is_tetra:
    r = input("FE_MEBT:FC_D1102 is in TetrA mode. Do you want me to switch to Pico mode? ")
    if r in ['yes','y','YES','Y']:
        caput('DIAG-RIO01:PICO_SWITCH2',0)
        time.sleep(5)
        is_tetra = caget('DIAG-RIO01:TETRAMM_ENABLED1')
        if is_tetra:
            r = input("FE_MEBT:FC_D1102 is still in TetrA mode. please check manually.")

##### check upstream apertures and attenuators

In [7]:
aperture_setPVs = ['FE_LEBT:AP_D0796:IN_CMD','FE_LEBT:AP_D0807:IN_CMD']
aperture_rdPVs  = ['FE_LEBT:AP_D0796:LMIN_RSTS','FE_LEBT:AP_D0807:LMIN_RSTS']
aperture_rd_targets = [1, 1]
aperture_rd_tols    = [0.1, 0.1]
isCMDsent = False
for i,pv in enumerate(aperture_rdPVs):
    pv_simple = pv.strip(':LMIN_RSTS').strip(':IN_CMD')
    target = aperture_rd_targets[i]
    val = caget(pv)
    tol = aperture_rd_tols[i]
    isin = target-tol < val < target+tol
    if not isin:
        r = input(f"aperture {pv_simple} is not inserted. Do you allow me to put it in? ")
        if r in ['yes','y','YES','Y']:
            caput(aperture_setPVs[i],target)
            print(f"aperture {pv_simple} is inserted")
if isCMDsent:
    time.sleep(3)
    for i,pv in enumerate(aperture_rdPVs):
        pv_simple = pv.strip(':LMIN_RSTS').strip(':IN_CMD')
        target = aperture_rd_targets[i]
        val = caget(pv)
        tol = aperture_rd_tols[i]
        isin = target-tol < val < target+tol
        if not isin:
            r = input(f"aperture {pv_simple} is still not inserted. please check manually. enter to continue")

##### check monitor PVs

# === Begin User Inputs ===

##### Define current at 100% transmission (FC814 reading)

In [9]:
maxBeamCurrent = 39.7

##### IO settings

In [10]:
data_acquire_rate = 5 # [Hz]
timespan_for_average = 2.0  # [sec]
additional_wait_after_powersupply_ramp  = 0.3 # [sec]

##### Optimizer setting 
Do not exceed budget 100. It will impractically slow down training and quering GP model

In [11]:
is_close_to_opt = True

if is_close_to_opt:
    n_init_budget       = 10   # recommended: about order of control parameters 
    n_global_opt_budget = 0
    n_local_opt_budget  = 50
    n_finetune_budget   = 8    # recommended: about order of control parameters 
else:
    n_init_budget       = 30          
    n_global_opt_budget = 30
    n_local_opt_budget  = 30
    n_finetune_budget   = 5    # recommended: less than number of control parameters 

_budget = n_init_budget +n_global_opt_budget +n_local_opt_budget +n_finetune_budget
_ramping_time = 2 # on average, each iter, considering polarity crossing time
_expected_oracle_time_cost = timespan_for_average + additional_wait_after_powersupply_ramp + _ramping_time
print(f"budget: {_budget}")
print(f"expected run time: {int(_budget*_expected_oracle_time_cost)} sec")

budget: 68
expected run time: 292 sec


### Define controls
to do list:
 - try a V-dipole instead of a V-corrector to avoid polarity crossing time
 - try solenoids 

In [12]:
control_CSETs= [
#     'FE_LEBT:PSC2_D0948:I_CSET', 'FE_LEBT:PSC1_D0948:I_CSET',
    'FE_LEBT:PSC2_D0964:I_CSET', 'FE_LEBT:PSC1_D0964:I_CSET',
    'FE_LEBT:PSC2_D0979:I_CSET', 'FE_LEBT:PSC1_D0979:I_CSET',
    'FE_LEBT:PSC2_D0992:I_CSET', 'FE_LEBT:PSC1_D0992:I_CSET',
]
control_RDs  = [pv.replace('_CSET','_RD') for pv in control_CSETs]

is_cryo = np.any(['_CB' in PV or '_CA' in PV or  '_CC' in PV or  '_CD' in PV or  '_CE' in PV for PV in control_CSETs])
if not is_close_to_opt and is_cryo:
    print('warn: global optimization is proposed with controls in croyo module. proceed with care')

In [13]:
x0 = caget_many(control_CSETs)

control_tols = []
control_min = []
control_max = []
for v, PV in zip(x0,control_CSETs):
    if 'PSC' in PV:
        control_min.append(v-0.5*AQ)
        control_max.append(v+0.5*AQ)
#         control_min.append( -0.8*AQ)
#         control_max.append( +0.8*AQ)
        control_tols.append(0.2)
    elif 'PSOL' in PV:
        control_min.append(0.9*v)
        control_max.append(1.1*v)
        control_tols.append(1.0)
    else:
        raise ValueError(f'control bounds for {PV} cannot be determined')

assert len(control_CSETs) == len(control_min) == len(control_max) == len(control_tols)
control_Lo_limit, control_Hi_limit = preset.get_limits(control_CSETs)
control_min = np.clip(control_min, a_min = control_Lo_limit, a_max = None)
control_max = np.clip(control_max, a_min = None, a_max = control_Hi_limit)
assert np.all(control_max > control_min)
control_bounds = torch.tensor([control_min.tolist(), control_max.tolist()], dtype=torch.float64)

print("============== check control bounds ================= ")
pd.DataFrame(np.array([x0,control_min,control_max,control_tols,control_Lo_limit,control_Hi_limit]).T,
             index=control_CSETs, 
             columns=['current value','control min','control max','tol','LoLim','HiLim'])

============== check control bounds ================= 


,current value,control min,control max,tol,LoLim,HiLim
FE_LEBT:PSC2_D0964:I_CSET,2.373,-0.843216,5.500000,0.2,-5.5,5.5
FE_LEBT:PSC1_D0964:I_CSET,-0.202,-3.418216,3.014216,0.2,-5.5,5.5
FE_LEBT:PSC2_D0979:I_CSET,0.481,-2.735216,3.697216,0.2,-5.5,5.5
FE_LEBT:PSC1_D0979:I_CSET,-0.624,-3.840216,2.592216,0.2,-5.5,5.5
FE_LEBT:PSC2_D0992:I_CSET,0.746,-2.470216,3.962216,0.2,-5.5,5.5
FE_LEBT:PSC1_D0992:I_CSET,-1.922,-5.138216,1.294216,0.2,-5.5,5.5


### Define objectives

In [16]:
BPM_snapshot_fname = '20260418_1152_238U37_17p3MeVu_2W_to_CSS_slitclosed.bpm'
if BPM_snapshot_fname is not None:
    objective_goal = preset.get_MEBT_objective_goal_from_BPMoverview(BPM_snapshot_fname)
    objective_goal['FE_MEBT:BCM_D1055:AVGPK_RD'] = {'more than': 0.99*maxBeamCurrent},
    objective_goal['FE_MEBT:FC_D1102:PKAVG_RD' ]  ={'more than':  0.8*maxBeamCurrent},
    display(objective_goal)

{'FE_MEBT:BPM_D1056:XPOS_RD': 0.25163805320315985,
 'FE_MEBT:BPM_D1056:YPOS_RD': 0.12572507640791447,
 'FE_MEBT:BPM_D1056:PHASE_RD': 76.78251733880323,
 'FE_MEBT:BPM_D1072:XPOS_RD': 0.04461801988247305,
 'FE_MEBT:BPM_D1072:YPOS_RD': 0.14918937946766403,
 'FE_MEBT:BPM_D1072:PHASE_RD': -26.69204704418517,
 'FE_MEBT:BPM_D1094:XPOS_RD': -0.00830562782880001,
 'FE_MEBT:BPM_D1094:YPOS_RD': 0.10088006089580875,
 'FE_MEBT:BPM_D1094:PHASE_RD': -15.958044148322497,
 'FE_MEBT:BCM_D1055:AVGPK_RD': ({'more than': 39.303000000000004},),
 'FE_MEBT:FC_D1102:PKAVG_RD': ({'more than': 31.760000000000005},)}

In [17]:
objective_goal= {
    'FE_MEBT:BPM_D1056:XPOS_RD': 0.0,
    'FE_MEBT:BPM_D1056:YPOS_RD': 0.0,
    'FE_MEBT:BPM_D1056:PHASE_RD': 76.78251733880323,
    'FE_MEBT:BPM_D1072:XPOS_RD': 0.04461801988247305,
    'FE_MEBT:BPM_D1072:YPOS_RD': 0.14918937946766403,
    'FE_MEBT:BPM_D1072:PHASE_RD': -26.69204704418517,
    'FE_MEBT:BPM_D1094:XPOS_RD': -0.00830562782880001,
    'FE_MEBT:BPM_D1094:YPOS_RD': 0.10088006089580875,
    'FE_MEBT:BPM_D1094:PHASE_RD': -15.958044148322497,
    'FE_MEBT:BCM_D1055:AVGPK_RD': {'more than': 0.99*maxBeamCurrent},
    'FE_MEBT:FC_D1102:PKAVG_RD' : {'more than':  0.8*maxBeamCurrent},
}

In [18]:
objective_tolerance= {
    'FE_MEBT:BPM_D1056:XPOS_RD' : 1,  
    'FE_MEBT:BPM_D1056:YPOS_RD' : 1,
    'FE_MEBT:BPM_D1056:PHASE_RD': 1,
    'FE_MEBT:BPM_D1072:XPOS_RD' : 1,
    'FE_MEBT:BPM_D1072:YPOS_RD' : 1,
    'FE_MEBT:BPM_D1072:PHASE_RD': 1,
    'FE_MEBT:BPM_D1094:XPOS_RD' : 1,
    'FE_MEBT:BPM_D1094:YPOS_RD' : 1,
    'FE_MEBT:BPM_D1094:PHASE_RD': 1,
    'FE_MEBT:BCM_D1055:AVGPK_RD': 0.02*maxBeamCurrent,
    'FE_MEBT:FC_D1102:PKAVG_RD' : 0.05*0.8*maxBeamCurrent,
}

In [19]:
##== If weight of any objective is zero, the corresponding objective will be removed.
objective_weight= {
    'FE_MEBT:BPM_D1056:XPOS_RD' : 0.8,
    'FE_MEBT:BPM_D1056:YPOS_RD' : 0.8,
    'FE_MEBT:BPM_D1056:PHASE_RD': 2.0,
    'FE_MEBT:BPM_D1072:XPOS_RD' : 0.0,
    'FE_MEBT:BPM_D1072:YPOS_RD' : 0.0,
    'FE_MEBT:BPM_D1072:PHASE_RD': 2.0,
    'FE_MEBT:BPM_D1094:XPOS_RD' : 0.0,
    'FE_MEBT:BPM_D1094:YPOS_RD' : 0.0,
    'FE_MEBT:BPM_D1094:PHASE_RD': 2.0,
    'FE_MEBT:BCM_D1055:AVGPK_RD': 1.0,
    'FE_MEBT:FC_D1102:PKAVG_RD' : 1.0,
}

In [20]:
objective_weight = {k:v for k,v in objective_weight.items() if v!=0}
objective_PVs = list(objective_weight.keys())
objective_goal = {pv:objective_goal[pv] for pv in objective_PVs}
objective_tolerance = {pv:objective_tolerance[pv] for pv in objective_PVs}

print("============== check objective ================= ")
print(" too small numbers may rounded to show 0 but actual value may not be 0")
pd.DataFrame([objective_goal,objective_tolerance,objective_weight],index=['goal','norm','weight']).T

============== check objective ================= 
 too small numbers may rounded to show 0 but actual value may not be 0


,goal,norm,weight
FE_MEBT:BPM_D1056:XPOS_RD,0.0,1.0,0.8
FE_MEBT:BPM_D1056:YPOS_RD,0.0,1.0,0.8
FE_MEBT:BPM_D1056:PHASE_RD,76.782517,1.0,2.0
FE_MEBT:BPM_D1072:PHASE_RD,-26.692047,1.0,2.0
FE_MEBT:BPM_D1094:PHASE_RD,-15.958044,1.0,2.0
FE_MEBT:BCM_D1055:AVGPK_RD,{'more than': 39.303000000000004},0.794,1.0
FE_MEBT:FC_D1102:PKAVG_RD,{'more than': 31.760000000000005},1.588,1.0


###### preprare live plot  --> specify PV groups (list of list) to plot togather

monitors

In [21]:
extra_monitors = []
monitor_PVs = objective_PVs + extra_monitors
CURRENTs = [pv for pv in monitor_PVs if '_RD' in pv and (':FC_D' in pv or ':BCM_D' in pv) ]
POSs = [pv for pv in monitor_PVs if ':BPM_D' in pv and 'POS_RD' in pv]
PHASEs = [pv for pv in monitor_PVs if 'PHASE' in pv]
monitor_groups = [CURRENTs,POSs,PHASEs]

controls

In [22]:
CORs = [pv for pv in control_CSETs if ':PSC' in pv]
SOLs = [pv for pv in control_CSETs if ':PSOL' in pv]
control_CSETs_groups = [CORs,SOLs]
monitor_groups = [l for l in monitor_groups if len(l)>0]

CORs = [pv for pv in control_RDs if ':PSC' in pv]
SOLs = [pv for pv in control_RDs if ':PSOL' in pv]
control_RDs_groups = [CORs,SOLs]

# === End of User Inputs ===

In [23]:
now0 = datetime.datetime.now()
fname = now0.strftime('%Y%m%d_%H%M')+'['+ion+'][pyBO][MEBT]FC1102'
fname

'20260706_1120[238U37][pyBO][MEBT]FC1102'

##### Prepare machine evaluator

In [24]:
_expected_bo_computation_time = 1 # [sec]
_run_async = 0.5 < abs(_expected_oracle_time_cost/_expected_bo_computation_time - 1) < 2
print(f"_run_async: {_run_async}")

machineIO = construct_machineIO(
    fetch_data_time_span = timespan_for_average,
    ensure_set_timewait_after_ramp = additional_wait_after_powersupply_ramp,
    sample_interval = 1/data_acquire_rate,
    test = False,
)

_run_async: False


In [25]:
composite_objective_name = 'composite_obj'
obj_func = SingleTaskObjectiveFunction(
    objective_PVs = objective_PVs,
    composite_objective_name = composite_objective_name,
    objective_goal = objective_goal,
    objective_weight = objective_weight,
    objective_tolerance = objective_tolerance,
    p_order = 2,
    apply_bilog = False
)

In [26]:
oracle_key_names = {
    'x':control_RDs,
    'y':composite_objective_name,
}

oracle =  OracleEvaluator(
    machineIO,
    control_CSETs= control_CSETs,
    control_RDs  = control_RDs,
    control_tols = control_tols,
    oracle_key_names = oracle_key_names,
    monitor_PVs  = monitor_PVs,
    df_manipulators = [obj_func.calculate_objectives_from_df],
)

##### preprare live plot

In [28]:
bo = BOController(oracle, bounds = control_bounds)

[11:21:06.600] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.3 ms.


In [29]:
monitor_live = live_monitor_plot(
    bo, oracle,
    monitor_groups       = [l for l in monitor_groups if len(l)>0], 
    control_CSETs_groups = [l for l in control_CSETs_groups if len(l)>0],
    control_RDs_groups   = [l for l in control_RDs_groups if len(l)>0],
)
history_live = live_history_plot(bo)
monitor_live.start()
history_live.start()

# run

In [30]:
print("initializatin...")
bo.initialize(budget=n_init_budget, local_init=is_close_to_opt)

print("global optimization...")
for _ in range(n_global_opt_budget):
    fresh_train = True #(not bool(_%2)) or len(bo.train_x) < bo.bounds.shape[1]*2
    bo.step(mode="global", acq_type="qEI", fresh_train=fresh_train, plot_acq=False, asynchro=_run_async)
    
print("local optimization...")
for _ in range(n_local_opt_budget):
    fresh_train = True #(not bool(_%2)) or len(bo.train_x) < bo.bounds.shape[1]*2
    bo.step(mode="local", acq_type="qEI", fresh_train=fresh_train, plot_acq=False, asynchro=_run_async)
    
print("fine tuning...")
for _ in range(n_finetune_budget):
    bo.step(mode="fine_tune", acq_type="qEI", fresh_train=True, plot_acq=False, asynchro=_run_async)
    plt.show()

initializatin...
[11:21:14.114] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[11:21:18.008] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[11:21:22.313] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[11:21:26.525] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[11:21:30.722] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[11:21:34.431] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[11:21:38.633] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[11:21:42.628] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[11:21:47.119] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
global optimization...
local optimization...
[11:21:51.434] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[11:21:56.112] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[11:22:01.917] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[1

In [31]:
now1 = datetime.datetime.now()
print(f"optimization took {(now1-now0).seconds} sec")

optimization took 471 sec


# Set to Best solution 

In [32]:
x_hist = [h["x"] for h in bo.history]
y_hist = [h["y"] for h in bo.history]
imax = np.argmax(y_hist)

orcle_dic = oracle(x_hist[imax])

print("Best x:", x_hist[imax])
print("Best y (old):", y_hist[imax])
print("Best y (new):", orcle_dic['y'])

[11:28:41.417] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
[11:29:01.427] WARNING: phantasy.~.epics_tools: Established 19 PVs in 0.1 ms.
Best x: [ 0.49272    -0.67215273  1.83404     0.64182     1.278      -0.17039   ]
Best y (old): [0.65362358]
Best y (new): [0.50675988]


### comparision: before and after opt

In [33]:
tmp = np.vstack((x_hist[0],x_hist[imax]))
pd.DataFrame(tmp,columns=control_CSETs,index=['before opt','after opt']).T

,before opt,after opt
FE_LEBT:PSC2_D0964:I_CSET,2.17182,0.492720
FE_LEBT:PSC1_D0964:I_CSET,-0.42217,-0.672153
FE_LEBT:PSC2_D0979:I_CSET,0.58285,1.834040
FE_LEBT:PSC1_D0979:I_CSET,-0.56380,0.641820
FE_LEBT:PSC2_D0992:I_CSET,0.52309,1.278000
FE_LEBT:PSC1_D0992:I_CSET,-1.75191,-0.170390
